In [1]:
import pandas as pd
import requests
import json
import re

In [ ]:
import os

os.environ['SARVAM_API_KEY'] = "insert_your_api_key_here"
API_KEY = os.environ['SARVAM_API_KEY']

URL = "https://api.sarvam.ai/v1/chat/completions"

In [3]:
SYSTEM_PROMPT_FULL = """You are a West Bengal electoral demographic classifier.
Given a voter name and their district, output ONLY a compact JSON object with these exact keys:

R: Religion — MUST be exactly one of: H, M, O
  H = Hindu   M = Muslim   O = Other (Sikh, Christian, Jain, Buddhist, etc.)
  U is NOT valid. You must pick the most likely option.

G: General category — MUST be exactly one of: GEN, SC, ST, OBC
  U is NOT valid. You must pick the most likely option.

SG: Religious subgroup — depends on R:
  If R=H: GUC (Brahmin/Kayastha/Baidya), MAT (Matua/Namasudra), RAJ (Rajbangshi),
          NAM (Namasudra other), MAH (Mahishya), SAN (Santhal), TEA (Tea Garden Worker),
          OBC_H (Hindu OBC), O (Other Hindu)
  If R=M: BM (Bengali Muslim), UM (Urdu-speaking Muslim), O (Other Muslim)
  If R=O: SG must be O
  U is NOT valid. Pick the most plausible subgroup.

CONF: H, M, or L

Return ONLY valid JSON. No explanation. Example:
{"R":"H","G":"SC","SG":"MAT","CONF":"M"}"""

In [4]:
def build_g_sg_system_prompt(r_code):
    religion_map = {"H": "Hindu", "M": "Muslim", "O": "neither Hindu nor Muslim"}
    religion_str = religion_map.get(r_code, "neither Hindu nor Muslim")

    if r_code == "H":
        sg_options = ("GUC (Brahmin/Kayastha/Baidya), MAT (Matua/Namasudra), RAJ (Rajbangshi), "
                      "NAM (Namasudra other), MAH (Mahishya), SAN (Santhal), TEA (Tea Garden Worker), "
                      "OBC_H (Hindu OBC)")
    elif r_code == "M":
        sg_options = "BM (Bengali Muslim), UM (Urdu-speaking Muslim)"
    else:
        sg_options = "O (only valid value when religion is Other)"

    return f"""You are a West Bengal electoral demographic classifier.
This voter is {religion_str}. Given their name and district, output ONLY a compact JSON object:

G: General category — MUST be exactly one of: GEN, SC, ST, OBC
  GEN = General/Forward   SC = Scheduled Caste   ST = Scheduled Tribe   OBC = Other Backward Class
  U is NOT valid. Pick the most likely option.

SG: Religious subgroup — MUST be one of: {sg_options}
  U is NOT valid. Pick the most plausible subgroup.

Return ONLY valid JSON. No explanation. Example:
{{"G":"SC","SG":"MAT"}}"""

In [5]:
def call_api(system_prompt, name, district=""):
    context = f"Name: {name}"
    if district:
        context += f"\nDistrict: {district}"

    data = {
        "model": "sarvam-m",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": context}
        ]
    }
    response = requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json=data
    )
    raw = response.json()["choices"][0]["message"]["content"].strip()
    cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"error": "parse_failed"}

In [ ]:
import pandas as pd
import os
import json
import re
import requests

# Load your dataframe (adjust path as needed)
df = pd.read_csv("rows_to_process_part4_progress.csv", low_memory=False)

# Define start and end rows using iloc
start_row = 9464
end_row = 10402  # Process first 28,500 rows

# Check for existing checkpoint
checkpoint_file = "part4_2_checkpoint.csv"
if os.path.exists(checkpoint_file):
    checkpoint_df = pd.read_csv(checkpoint_file)
    start_row = len(checkpoint_df)
    print(f"Found checkpoint: Already processed {start_row} rows")
    print(f"Resuming from row {start_row}")

print(f"Processing rows {start_row} to {end_row}")

k = start_row
for idx, row in df.iloc[start_row:end_row].iterrows():
    # Get R value from existing data
    r_value = str(row["R"]) if pd.notna(row["R"]) and row["R"] != "U" else ""
    
    # Build system prompt based on R
    if r_value:
        system_prompt = build_g_sg_system_prompt(r_value)
    else:
        # If no R, use a generic prompt (though this shouldn't happen for your mask)
        system_prompt = build_g_sg_system_prompt("H")  # Default to Hindu
    
    # Call API to get G and SG
    result = call_api(
        system_prompt,
        str(row["name"]),
        str(row["district"]) if pd.notna(row["district"]) else ""
    )
    
    if "error" not in result:
        # Update G and SG only
        if result.get("G"):
            df.at[idx, "G"] = result.get("G")
        if result.get("SG"):
            df.at[idx, "SG"] = result.get("SG")
    
    print(f"{k + 1}/{end_row} - Index {idx}: {result}")
    print("done", k)
    k += 1
    
    # Save checkpoint every 100 rows
    if k % 100 == 0:
        # Save the full dataframe
        df.to_csv("rows_to_process_part4_2_progress.csv", index=False)
        
        # Save checkpoint tracking
        checkpoint_df = df.iloc[:k].copy()
        checkpoint_df.to_csv(checkpoint_file, index=False)
        print(f"  ✓ Checkpoint saved at {k} rows")


In [ ]:
df = pd.read_csv("rows_to_process_progress.csv", low_memory=False)
df.head(10)

In [11]:
df.to_csv("updated.csv", index=False)